In [6]:
!pip install pypdf

     ---------------------------------------- 0.0/333.7 kB ? eta -:--:--
     -------------------------------------- 333.7/333.7 kB 6.9 MB/s eta 0:00:00



[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [9]:
!pip install langchain-pinecone

     ---------------------------------------- 0.0/71.9 kB ? eta -:--:--
     ---------------------------------------- 0.0/71.9 kB ? eta -:--:--
     ---------- --------------------------- 20.5/71.9 kB 320.0 kB/s eta 0:00:01
     -------------------------------------  71.7/71.9 kB 653.6 kB/s eta 0:00:01
     -------------------------------------- 71.9/71.9 kB 560.7 kB/s eta 0:00:00
   ---------------------------------------- 0.0/87.2 kB ? eta -:--:--
   ---------------------------------------- 87.2/87.2 kB 4.8 MB/s eta 0:00:00



[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
# from pypdf import PdfReader

In [10]:
# check package varsion
import importlib.metadata

# List the distribution package names
packages = ["langchain", "langchain-community", "langchain-openai", 
            "pinecone", "langchain-pinecone",  "pypdf"]

for package in packages:
    try:
        version = importlib.metadata.version(package)
        print(f"{package} version: {version}")
    except importlib.metadata.PackageNotFoundError:
        print(f"{package} is not installed in this environment.")


langchain version: 1.3.6
langchain-community version: 0.4.2
langchain-openai version: 1.2.2
pinecone version: 7.3.0
langchain-pinecone version: 0.2.13
pypdf version: 6.10.2


# Put below code in  rag_core.py

In [17]:
# rag_core.py  —  Pinecone edition

import os
import tempfile
import time

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_pinecone import PineconeVectorStore          

from pinecone import Pinecone, ServerlessSpec               # pinecone>=5


# -------- LOAD + SPLIT --------
def load_and_split_pdfs(files):
    """
    Accept a list of Streamlit UploadedFile objects or local binary file handles.
    Returns a flat list of LangChain Document chunks.
    """
    all_docs = []

    for file in files:
        if hasattr(file, "read"):
            # Streamlit / binary file handle → write to a temp file first
            with tempfile.NamedTemporaryFile(delete=False, suffix=".pdf") as tmp:
                tmp.write(file.read())
                file_path = tmp.name
            owns_tmp = True
        else:
            file_path = file
            owns_tmp = False

        try:
            loader = PyPDFLoader(file_path)
            docs = loader.load()
        finally:
            if owns_tmp:
                os.unlink(file_path)        # clean up temp file immediately

        splitter = RecursiveCharacterTextSplitter(
            chunk_size=100,
            chunk_overlap=10,
        )
        all_docs.extend(splitter.split_documents(docs))

    return all_docs


# -------- BUILD / CONNECT VECTOR STORE --------
def index_is_populated(pc, index_name):
    """
    Returns True if the Pinecone index exists AND already contains vectors.
    Used to skip re-ingestion on subsequent runs.
    """
    existing = [idx.name for idx in pc.list_indexes()]
    if index_name not in existing:
        return False
    stats = pc.Index(index_name).describe_index_stats()
    return stats.get("total_vector_count", 0) > 0


def build_vector_store(docs, openai_api_key, pinecone_api_key, index_name="rag-index"):
    """
    Upserts document chunks into a Pinecone serverless index.

    - If the index does not exist → creates it, then upserts.
    - If the index exists but is EMPTY → upserts.
    - If the index exists and already has vectors → skips upsert entirely
      and just returns a connected PineconeVectorStore.

    Args:
        docs            : List[Document] from load_and_split_pdfs()
        openai_api_key  : str
        pinecone_api_key: str
        index_name      : str  (default: 'rag-index')

    Returns:
        PineconeVectorStore instance
    """
    embeddings = OpenAIEmbeddings(
        model="text-embedding-3-small",        # 1536-dim, fast & cheap
        openai_api_key=openai_api_key,
    )

    pc = Pinecone(api_key=pinecone_api_key)

    # ---- Create index if it doesn't exist ----
    existing = [idx.name for idx in pc.list_indexes()]
    if index_name not in existing:
        print(f"  Creating Pinecone index '{index_name}' ...")
        pc.create_index(
            name=index_name,
            dimension=1536,
            metric="cosine",
            spec=ServerlessSpec(
                cloud="aws",                   # change to "gcp" / "azure" if needed
                region="us-east-1",
            ),
        )
        while not pc.describe_index(index_name).status["ready"]:
            print("  Waiting for index to be ready...")
            time.sleep(2)
        print("  Index ready.")

    # ---- Skip upsert if vectors are already present ----
    if index_is_populated(pc, index_name):
        print(f"  Index '{index_name}' already populated — skipping upsert.")
        return PineconeVectorStore(
            index_name=index_name,
            embedding=embeddings,
            pinecone_api_key=pinecone_api_key,
        )

    # ---- First-time upsert ----
    print(f"  Upserting {len(docs)} chunks into '{index_name}' ...")
    vectorstore = PineconeVectorStore.from_documents(
        documents=docs,
        embedding=embeddings,
        index_name=index_name,
        pinecone_api_key=pinecone_api_key,
    )
    print("  Upsert complete.")
    return vectorstore


def load_vector_store(openai_api_key, pinecone_api_key, index_name="rag-index"):
    """
    Connect to an *existing* Pinecone index without re-uploading documents.
    Use this after the first run so you don't pay for redundant upserts.
    """
    embeddings = OpenAIEmbeddings(
        model="text-embedding-3-small",
        openai_api_key=openai_api_key,
    )
    return PineconeVectorStore(
        index_name=index_name,
        embedding=embeddings,
        pinecone_api_key=pinecone_api_key,
    )


# -------- RETRIEVER --------
def get_retriever(vectorstore, k=3):
    return vectorstore.as_retriever(search_kwargs={"k": k})


# -------- ASK LLM --------
def ask_llm(question, retriever, openai_api_key):
    llm = ChatOpenAI(
        model="gpt-4o-mini",
        temperature=0,
        openai_api_key=openai_api_key,
    )

    docs = retriever.invoke(question)
    context = "\n\n".join([doc.page_content for doc in docs])

    prompt = f"""Answer using the context below. Keep it short.

            Context:
            {context}
            
            Question: {question}
            """

    response = llm.invoke(prompt)
    return response.content, docs

In [18]:
# -------- MAIN (local test) --------
if __name__ == "__main__":
    from dotenv import load_dotenv
    load_dotenv()

    OPENAI_API_KEY   = os.getenv("OPENAI_API_KEY")
    PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
    INDEX_NAME       = "rag-index"
    test_pdf         = "document-loxford-company.pdf"

    if not os.path.exists(test_pdf):
        print("PDF file not found.")
        exit(1)

    #  Smart ingestion:                                                    #
    #  - Checks whether the Pinecone index already has vectors.           #
    #  - If yes  → skips PDF loading & upsert entirely (fast path).       #
    #  - If no   → loads, splits, and upserts once, then never again.     #
    pc = Pinecone(api_key=PINECONE_API_KEY)

    if index_is_populated(pc, INDEX_NAME):
        print(f"Index '{INDEX_NAME}' already exists with data. Connecting directly...")
        vectorstore = load_vector_store(OPENAI_API_KEY, PINECONE_API_KEY, INDEX_NAME)
    else:
        print("First run detected — loading and ingesting PDF...")
        with open(test_pdf, "rb") as fh:
            docs = load_and_split_pdfs([fh])
        print(f"  Total chunks: {len(docs)}")
        vectorstore = build_vector_store(
            docs,
            openai_api_key=OPENAI_API_KEY,
            pinecone_api_key=PINECONE_API_KEY,
            index_name=INDEX_NAME,
        )

    retriever = get_retriever(vectorstore)

    # ------------------------------------------------------------------ #
    #  Interactive Q&A loop — type 'exit' or 'quit' to stop.             #
    # ------------------------------------------------------------------ #
    print("\n" + "="*50)
    print("  RAG Chat ready. Type 'exit' to quit.")
    print("="*50)

    while True:
        question = input("\nYour question: ").strip()

        if not question:
            continue

        if question.lower() in {"exit", "quit", "q"}:
            print("Goodbye!")
            break

        answer, retrieved_docs = ask_llm(question, retriever, OPENAI_API_KEY)

        print("\n--- Retrieved Chunks ---")
        for i, d in enumerate(retrieved_docs):
            print(f"\n[Chunk {i+1}]")
            print(d.page_content)

        print("\n--- Answer ---")
        print(answer)

First run detected — loading and ingesting PDF...
  Total chunks: 37
  Creating Pinecone index 'rag-index' ...
  Index ready.
  Upserting 37 chunks into 'rag-index' ...
  Upsert complete.

  RAG Chat ready. Type 'exit' to quit.



Your question:  Who is the CTO of the company



--- Retrieved Chunks ---

[Chunk 1]
AI research and real-world business applications. 
• CEO: Daniel Reeves 
• CTO: Anika Lomax

[Chunk 2]
The company is headquartered in Austin, Texas, USA, with additional offices in: 
• London, UK

[Chunk 3]
Loxford Technologies – Company Overview

--- Answer ---
The CTO of the company is Anika Lomax.



Your question:  How many people are working in the company



--- Retrieved Chunks ---

[Chunk 1]
employs over 850 professionals, including engineers, data scientists, and 
consultants.

[Chunk 2]
The company is headquartered in Austin, Texas, USA, with additional offices in: 
• London, UK

[Chunk 3]
million, with a year-over-year growth rate of approximately 22%. The company

--- Answer ---
The company employs over 850 professionals.



Your question:  What is this document about?



--- Retrieved Chunks ---

[Chunk 1]
Recent Developments

[Chunk 2]
Mission: 
To empower organizations with intelligent systems that enhance decision-making

[Chunk 3]
Enterprise-grade Retrieval-Augmented Generation systems for internal 
knowledge management.

--- Answer ---
The document is about empowering organizations through intelligent systems that improve decision-making, specifically focusing on enterprise-grade Retrieval-Augmented Generation systems for internal knowledge management.



Your question:  q


Goodbye!


# Sample run:
```
First run detected — loading and ingesting PDF...
  Total chunks: 37
  Creating Pinecone index 'rag-index' ...
  Index ready.
  Upserting 37 chunks into 'rag-index' ...
  Upsert complete.

==================================================
  RAG Chat ready. Type 'exit' to quit.
==================================================

Your question:  Who is the CTO of the company

--- Retrieved Chunks ---

[Chunk 1]
AI research and real-world business applications. 
• CEO: Daniel Reeves 
• CTO: Anika Lomax

[Chunk 2]
The company is headquartered in Austin, Texas, USA, with additional offices in: 
• London, UK

[Chunk 3]
Loxford Technologies – Company Overview

--- Answer ---
The CTO of the company is Anika Lomax.

Your question:  How many people are working in the company

--- Retrieved Chunks ---

[Chunk 1]
employs over 850 professionals, including engineers, data scientists, and 
consultants.

[Chunk 2]
The company is headquartered in Austin, Texas, USA, with additional offices in: 
• London, UK

[Chunk 3]
million, with a year-over-year growth rate of approximately 22%. The company

--- Answer ---
The company employs over 850 professionals.

Your question:  What is this document about?

--- Retrieved Chunks ---

[Chunk 1]
Recent Developments

[Chunk 2]
Mission: 
To empower organizations with intelligent systems that enhance decision-making

[Chunk 3]
Enterprise-grade Retrieval-Augmented Generation systems for internal 
knowledge management.

--- Answer ---
The document is about empowering organizations through intelligent systems that improve decision-making, specifically focusing on enterprise-grade Retrieval-Augmented Generation systems for internal knowledge management.

Your question:  q
Goodbye!

# Sample question
1) What is this document about?
2) Who is the CTO of the company ?
3) How many people are working in the company ?
